In [47]:
pip install geemap

In [48]:
import geemap
import ee
import os
import pandas as pd # les bases de données
import numpy as np #il gére aussi des bases de données
import matplotlib.pyplot as plt # permet de tracer des diagrammmes
import seaborn as sns

In [49]:
ee.Authenticate() #Cette ligne sert à prouver votre identité auprès de Google
ee.Initialize(project='initiation-gee') #Vous vous installez à un bureau spécifique (votre projet) pour commencer à ouvrir les livres et travailler
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map

In [ ]:
roi = ee.FeatureCollection('projects/initiation-gee/assets/roi_rdt_niakhar').geometry()

Map.centerObject(roi, 10)

In [ ]:
#implementer sentinel 2
s2 = ee.ImageCollection("COPERNICUS/S2_HARMONIZED")  #sentinel L2A
filtered = s2 \
  .filter(ee.Filter.date('2019-01-01', '2019-03-30')) \
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)) \

def maskS2clouds(image):
  qa = image.select('QA60')  #bande pour masquer les cirrus et nuage
  cloudBitMask = 1 << 10
  cirrusBitMask = 1 << 11
  mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
             qa.bitwiseAnd(cirrusBitMask).eq(0))
  return image.updateMask(mask).multiply(0.0001) \
      .select("B.*") \
      .copyProperties(image, ["system:time_start"])

filtered = filtered.map(maskS2clouds)

composite  = filtered.mean()
composite = composite.clip(roi)
Map.addLayer(composite, {'bands': ['B11', 'B8', 'B4'], 'min':0.1, 'max':0.4 }, 'RGB')   #composer colorer en RGB
Map

In [52]:
###
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('ndvi')
ndvi = ndvi.clip(roi)

ndvi_vis = {
    'min': 0.1,
    'max': 0.2,
    'palette': ['blue', 'white', 'green']
}

ndre_vis = composite.expression(
'(NIR-RE/NIR+RE)', {
    'NIR': composite.clip(roi).select('B8'),
    'RE': composite.clip(roi).select('B5')
}).rename('ndre')
ndre = ndre_vis.clip(roi)


# Map.addLayer(ndvi)
# Map

In [ ]:
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('ndvi')
ndvi = ndvi.clip(roi)
ndwi = composite.normalizedDifference(['B3', 'B8']).rename('ndwi')
ndwi = ndwi.clip(roi)


Msavi = composite.expression(
    '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2',
    {
        'NIR': composite.select('B8'),
        'RED': composite.select('B4')
    }
).rename('msavi')
Msavi = Msavi.clip(roi)


Osavi = composite.expression(
    '(1.16 * (NIR - RED)) / (NIR + RED + 0.16)',
    {
        'NIR': composite.select('B8'),
        'RED': composite.select('B4')
    }
).rename('osavi')
Osavi = Osavi.clip(roi)

In [ ]:
carte_terrain = ee.FeatureCollection ('projects/initiation-gee/assets/Rdt_mil_Niakhar_2019_jean')

In [55]:
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B11', 'B8A', 'B12', 'ndvi', 'ndwi', 'msavi', 'osavi']
####stack d'iamage
image = composite.addBands(ndvi).addBands(ndwi).addBands(Msavi).addBands(Osavi)
image = image.select(bands)

In [56]:
geemap.ee_export_image(image.clip(roi), filename = 'image_dj.tif', scale = 100, region = roi, file_per_band=False)

Generating URL ...
Please wait ...
Data downloaded to /content/image_dj.tif


In [57]:
label = 'Rdt_s'
sample = image.select(bands).sampleRegions(
    **{'collection': carte_terrain, 'properties': [label], 'scale': 10
    }
)

# Adds a column of deterministic pseudorandom numbers.
#sample = sample.randomColumn()#sample = échantillons


split = 0.8
training = sample.filter(ee.Filter.lt('random', split))
validation = sample.filter(ee.Filter.gte('random', split))


In [58]:
geemap.ee_to_csv(sample, "sample.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt  # les graphes
import seaborn as sns


In [60]:
sample_path =  "/content/sample.csv" # chemin de l'échantillon
sample_path = pd.read_csv(sample_path)
sample_path

,B1,B11,B12,B2,B3,B4,B5,B6,B7,B8A,Rdt_s,msavi,ndvi,ndwi,osavi
0,0.161008,0.502038,0.423885,0.151800,0.161692,0.217162,0.234900,0.257446,0.283931,0.324754,1430.2,0.092137,0.135556,-0.276482,0.119265
1,0.161008,0.488677,0.409000,0.149546,0.159815,0.215085,0.227669,0.251385,0.277000,0.315954,1430.2,0.087716,0.130458,-0.272638,0.114348
2,0.161008,0.488677,0.409000,0.146823,0.157500,0.210385,0.227669,0.251385,0.277000,0.315954,1430.2,0.086466,0.130490,-0.269194,0.113756
3,0.164031,0.512692,0.433908,0.159262,0.172438,0.238054,0.250385,0.278862,0.310769,0.348915,1430.2,0.097774,0.135595,-0.289173,0.121884
4,0.161008,0.519023,0.441269,0.157915,0.171008,0.234669,0.249462,0.274777,0.303500,0.343354,1430.2,0.095622,0.134000,-0.284935,0.120011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4694,0.168921,0.498371,0.400529,0.157779,0.175693,0.240914,0.266500,0.290721,0.322900,0.364521,652.0,0.102179,0.140107,-0.290294,0.126425
4695,0.168921,0.498371,0.400529,0.156957,0.174829,0.239300,0.266500,0.290721,0.322900,0.364521,652.0,0.101773,0.140137,-0.289500,0.126263
4696,0.168921,0.490421,0.391043,0.157229,0.174000,0.240057,0.255557,0.285486,0.319814,0.362793,652.0,0.104859,0.143669,-0.296411,0.129655
4697,0.168921,0.490421,0.391043,0.157579,0.174707,0.241579,0.255557,0.285486,0.319814,0.362793,652.0,0.105322,0.143729,-0.297498,0.129893


In [61]:
#select one column
y = sample_path[['Rdt_s']]
x = sample_path[['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B11', 'B8A', 'B12', 'ndvi', 'ndwi', 'msavi', 'osavi']]



In [62]:


#split the data

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.5, random_state=42)
#upsampling




#x_train, y_train = smote.fit_resample(x_train, y_train)

#normalize data
#from sklearn.preprocessing import StandardScaler
#sc = StandardScaler()
#x_train = sc.fit_transform(x_train)
#x_test = sc.transform(x_test)
#shape
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)



(2349, 14)
(2349, 1)
(2350, 14)
(2350, 1)


In [63]:
#import other models from sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
#import other models regression from sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression


In [64]:
#model regression random forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_hastie_10_2
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt
from sklearn.metrics import r2_score
#impor SVR
from sklearn.svm import SVR
#model = LinearRegression()
#model
model = RandomForestRegressor(n_estimators=200, random_state=42)

#model = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=.1)
#imprt decision tree for regression
#from sklearn.tree import DecisionTreeRegressor
#model = DecisionTreeRegressor(random_state=42)
#model = GradientBoostingRegressor(n_estimators=200, random_state=42)
#model = AdaBoostRegressor(n_estimators=200, random_state=42, learning_rate=0.1, loss='linear')

# select hyperparameters
n_estimators = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
max_depth = [5, 8, 15, 25, 30, None]
min_samples_split = [1, 2, 5, 10, 15, 100]
min_samples_leaf = [1, 2, 5, 10, 15]
max_features = [None, 'sqrt', 'log2']
bootstrap = [True, False]
algorithm = ['SAMME', 'SAMME.R'],
learning_rate = [0.1, 0.01, 0.001, 0.0001]
#oob_score = [True, False]

hyperF = dict(n_estimators=n_estimators,  learning_rate=learning_rate)


# When having limitted resources: conduct a Randomized Search CV
from sklearn.model_selection import RandomizedSearchCV

randomF = RandomizedSearchCV(model, hyperF, n_iter=10, n_jobs=-1, cv=10, verbose=1, random_state=42)

# Train Random Forest Model
model = model.fit(x_train, y_train)
#cross_val_score(estimator = model, X = x_train, y = y_train, cv = 10)
y_pred = model.predict(x_test)
r2 = r2_score(y_test, y_pred)
#r2 percentage
r2 = r2*100
print('R2: {:.2f} %'.format(r2))
#print("The mean accuracy of the model is:", model.score(test_features, test_labels))




R2: 79.54 %


In [ ]:

#model = model.fit(x_train, y_train)
#predict
y_pred = model.predict(x_test)
#K-fold cross validation
#rmse

rmse = sqrt(mean_squared_error(y_test, y_pred))
print('RMSE: {:.2f} Kg/ha'.format(rmse))
#r2
r2 = r2_score(y_test, y_pred)
#r2 percentage
r2 = r2*100

print('R2: {:.2f} %'.format(r2))

#MAE
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_pred)
print('MAE: {:.2f} Kg/ha'.format(mae))
#equation



In [ ]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(y_test, y_pred)
# equation y = ax + b

a = regressor.coef_
b = regressor.intercept_
print('y = {:.2f}x + {:.2f}'.format(a[0], b))


In [ ]:

#plot
plt.figure(figsize=(10,5))
plt.scatter(y_test, y_pred,s=5,c = 'green', alpha=0.5,   label='yield_gap')
plt.xlabel('Actual')
plt.xlim (0, 3500)
plt.ylabel('Predicted')
plt.ylim (0, 3500)
plt.title('yield_gap_regression_ADB_model_optimized')
#x =y line
plt.plot(y_test, y_test, color='red', linewidth=1, label='1st_bissector', linestyle='dashdot', alpha=0.5)

#add r2 and rmse to the plot
plt.text(2500, 400, 'R2 = {:.2f} %'.format(r2))
plt.text(2200, 1000, 'RMSE = {:.2f} Kg/ha'.format(rmse))
plt.text(2200, 200, 'MAE = {:.2f} Kg/ha'.format(mae))
plt.text(2200, 0, 'y = {:.2f}x + {:.2f}'.format(a[0], b), fontsize=10, color='black', alpha=0.5, fontstyle='italic', fontweight='bold',)
#fleche du nord

#compute regression line
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(y_test, y_pred)
y_pred1 = regressor.predict(y_test)
#add equation on plot
y_pred1_df = pd.DataFrame(y_pred1)
#y_pred1_df =y_pred1_df.sample(1000)
#y_test_df = y_test.sample(1000)
plt.plot(y_test, y_pred1, color='black', linewidth=1, label='regression line', linestyle='--', alpha=1)
#regression equation
plt.legend()
#add grid
plt.grid()
plt.show





In [ ]:
#feature importance
#RandomizedSearchCV

importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
#plot
plt.figure(figsize=(10,5))
plt.title('Feature Importances_ADB_model_optimized')
sns.barplot(x=importances[indices], y=x.columns[indices], palette="cividis", alpha=0.8, orient='h', edgecolor='black', linewidth=1, saturation=0.4, errcolor='black', errwidth=1)
plt.xlabel('Importance')
plt.ylabel('Features')
#color arrières plan
plt.grid()
plt.show()

In [ ]:
y_pred1 = y_pred.reshape(1069, 1)
#residual plot
plt.figure(figsize=(10,5))
#barplot

#barplot of residuals
plt.figure(figsize=(10,5))
plt.hist(y_test - y_pred1, bins=30, color='green', alpha=0.5,  label='residuals', edgecolor='black', linewidth=1.2, histtype='barstacked')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residuals distribution')
plt.grid()
plt.legend()
#95% confidence interval
plt.axvline(np.percentile(y_test - y_pred1, 2.5), color='red', linestyle='--', linewidth=1)
plt.axvline(np.percentile(y_test - y_pred1, 97.5), color='red', linestyle='--', linewidth=1)



plt.show()



In [ ]:
#convert to dataframe
y_test = pd.DataFrame(y_test)
y_pred = pd.DataFrame(y_pred)
#merge the two dataframes
#create a new dataframe
#residuals
residuals = y_test - y_pred
#drop NaN value

In [ ]:
#learning curve
from sklearn.model_selection import learning_curve
train_sizes, train_scores, test_scores = learning_curve(model, x_train, y_train, cv=10, n_jobs=-1, train_sizes=np.linspace(0.01, 1.0, 50), verbose=1)
#mean train and test scores
train_scores_mean = np.mean(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
#Calculate mean and std for error rate
train_scores_std = np.std(train_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)
#plot
plt.figure(figsize=(10,5))

plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std,\
                    alpha = 0.1, color = "magenta")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std,\
                    alpha = 0.1, color = "indigo")
plt.plot(train_sizes, train_scores_mean, "--", color="magenta", label='training_score')
plt.plot(train_sizes, test_scores_mean,"--", color="indigo", label='cross_validation_score')
plt.xlabel('train_sizes_exemples')
plt.ylabel('scores')
plt.grid()


plt.title('learning_curve_baseline_model')
plt.legend(loc = "best")
plt.grid()
plt.show()

